In [20]:
!pip install qiskit qiskit_aer qiskit_ibm_runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.4 MB/s eta 0:00:00


In [26]:
"""
DLP Adaptive Routing — Deterministic Noise Crossing
====================================================

Faithful implementation of the paper's Experiment 6a (Sec 4.6.2):
a 2-route softmax DLP router over mutually exclusive GHZ paths,
optimised via measurement-driven gradient updates.

Design choices aligned with the paper:
  - Two valid routes only (Path A, Path B). No partial/empty circuits.
  - Softmax router: p = softmax(lambda_A, lambda_B).  (Eq. 23)
  - DLP loss: L = 1 - sum_i p_i * F_i + w_hw * sum_i p_i * c_i.  (Eq. 24 + cost)
  - Gradient: dL/dlambda_i = p_i * (L_i - L_bar).  (standard softmax policy gradient)
  - Both routes measured every cycle (paper: "512 shots each").
  - One gradient step per cycle from fresh measurements.
  - Logits initialised to zero — no prior path preference.
  - Logit clamping to [-C, C] preserves recovery capability.

Noise isolation:
  - Noiseless simulator (only the injected Rx matters).
  - Deterministic linear crossing schedule, no jitter.
  - 8192 shots to suppress measurement variance.

Shot budget:
  - DLP: 2 circuits/cycle (both routes).
  - Static: reuses Path A measurement from DLP.
  - Greedy oracle: reuses both measurements from DLP.
  All three use the SAME 2 measurements — no inflated baselines.

Usage:
    python b_experiment9_fix_crossing.py
    IBM_API_KEY=... python b_experiment9_fix_crossing.py
"""

import numpy as np, math, os, time, json, traceback
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
from qiskit import QuantumCircuit, transpile
from qiskit.qasm2 import dumps as qasm2_dumps

# ============================================================
#  Figure style
# ============================================================
COLW = 3.5
DBLW = 7.0
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.labelsize': 10, 'axes.titlesize': 10,
    'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'figure.dpi': 300, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
    'lines.linewidth': 1.2, 'lines.markersize': 4,
    'axes.linewidth': 0.6, 'grid.linewidth': 0.4,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
})

OUT = 'figures/b_experiment9_fix_crossing/'
os.makedirs(OUT, exist_ok=True)

# ============================================================
#  API key (env-only, no file scanning, no printing)
# ============================================================
def load_api_key():
    return os.environ.get('IBM_API_KEY')

# ============================================================
#  Runners
# ============================================================
class IBMHardwareRunner:
    def __init__(self, api_key, backend_name='ibm_fez'):
        from qiskit_ibm_runtime import QiskitRuntimeService
        self.service = QiskitRuntimeService(
            channel="ibm_quantum_platform", token=api_key)
        self.backend = self.service.backend(backend_name)
        if not self.backend.status().operational:
            raise RuntimeError(f"{backend_name} not operational")
        from qiskit.transpiler.preset_passmanagers import (
            generate_preset_pass_manager)
        self.pm = generate_preset_pass_manager(
            optimization_level=1, target=self.backend.target)
        self.name = self.backend.name

    def run(self, qc, shots=512):
        from qiskit_ibm_runtime import SamplerV2 as Sampler
        transpiled = self.pm.run(qc)
        sampler = Sampler(self.backend)
        job = sampler.run([transpiled], shots=shots)
        pub = job.result()[0]; data = pub.data
        for attr in ['meas', 'c']:
            if hasattr(data, attr):
                return getattr(data, attr).get_counts()
        return getattr(data, list(vars(data).keys())[0]).get_counts()


class NoiselessAerRunner:
    """Ideal simulator — zero gate error."""
    def __init__(self):
        from qiskit_aer import AerSimulator
        self.backend = AerSimulator()
        self.name = 'aer_noiseless'

    def run(self, qc, shots=8192):
        tc = transpile(qc, self.backend)
        return self.backend.run(tc, shots=shots).result().get_counts()

# ============================================================
#  Circuit builders — two valid routes only
# ============================================================
def build_ghz_path_a(noise_angle):
    """Path A: Ry(pi/2,0) -> cx(0,1) -> cx(1,2), group-A noise."""
    qc = QuantumCircuit(3, 3)
    qc.ry(math.pi / 2, 0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    if abs(noise_angle) > 1e-8:
        qc.rx(noise_angle, 0)
        qc.rx(noise_angle, 1)
        qc.rx(noise_angle, 2)
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc


def build_ghz_path_b(noise_angle):
    """Path B: Ry(pi/2,0) -> cx(0,2) -> cx(2,1), group-B noise."""
    qc = QuantumCircuit(3, 3)
    qc.ry(math.pi / 2, 0)
    qc.cx(0, 2)
    qc.cx(2, 1)
    if abs(noise_angle) > 1e-8:
        qc.rx(noise_angle, 0)
        qc.rx(noise_angle, 1)
        qc.rx(noise_angle, 2)
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc


def ghz_fid(counts, shots):
    """GHZ fidelity = P(|000>) + P(|111>)."""
    return (counts.get('000', 0) + counts.get('111', 0)) / shots

# ============================================================
#  DLP Softmax Router  (Sec 4.6.1, Eqs 23-24)
# ============================================================
class DLPRouter:
    """
    Two-route softmax router with DLP-style differentiable loss.

    Structural logits:  lambda = (lambda_A, lambda_B) in R^2
    Path probabilities: p_i = exp(lambda_i) / sum_j exp(lambda_j)   (Eq. 23)

    Loss per cycle (Eq. 24, extended with hardware cost):
        L = sum_i p_i * [ (1 - F_i) + w_hw * c_i ]
          = expected infidelity + weighted hardware cost

    Gradient (softmax policy gradient):
        dL/dlambda_i = p_i * (L_i - L)
        where L_i = (1 - F_i) + w_hw * c_i  is the per-route loss.

    This pushes probability toward whichever route has lower total
    loss (higher fidelity, lower cost).
    """

    def __init__(self, init_logit=0.0, clamp=2.0,
                 w_hw=0.0, costs=None, lr=0.5):
        self.logits = np.array([init_logit, init_logit],
                               dtype=np.float64)
        self.clamp = clamp
        self.w_hw = w_hw
        self.costs = np.array(costs or [1.0, 1.0])
        # Adam state
        self.lr = lr
        self.m = np.zeros(2)          # first moment
        self.v = np.zeros(2)          # second moment
        self.t = 0                    # step counter
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.eps = 1e-8

    def probs(self):
        """Softmax probabilities."""
        e = np.exp(self.logits - self.logits.max())
        return e / e.sum()

    def selected_path(self):
        return int(np.argmax(self.logits))

    def update(self, fidelities):
        """
        Single Adam step from measured fidelities.
        """
        p = self.probs()
        F = np.array(fidelities)

        # Per-route loss: infidelity + hardware cost
        L_per = (1.0 - F) + self.w_hw * self.costs
        L_bar = (p * L_per).sum()

        # Softmax policy gradient: dL/dlambda_i = p_i * (L_i - L_bar)
        grad = p * (L_per - L_bar)

        # Adam update
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad ** 2
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)

        self.logits -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        self.logits = np.clip(self.logits, -self.clamp, self.clamp)

        return L_bar

# ============================================================
#  Deterministic noise profile
# ============================================================
def deterministic_crossing(num_cycles=12):
    """
    Smooth deterministic crossing. No random jitter.
    Group A: 0 -> 1.3 rad  (degrades)
    Group B: 1.0 -> 0 rad  (improves)
    """
    noise_A = np.linspace(0.0, 1.3, num_cycles)
    noise_B = np.linspace(1.0, 0.0, num_cycles)
    return noise_A, noise_B

# ============================================================
#  Main experiment
# ============================================================
def run_experiment(runner, noise_A, noise_B, num_cycles,
                   shots=8192, lr=0.5, n_grad_steps=10, w_hw=0.0):
    """
    Per cycle:
      1. Read current router state (BEFORE measuring).
      2. Measure BOTH routes (2 circuits, same as the paper).
      3. Deploy argmax route -> DLP fidelity.
      4. n_grad_steps gradient steps from the two measurements.
         (The softmax probabilities change between steps, so each
          gradient is recomputed — this is NOT stale reuse.)
      5. Baselines reuse the same measurements — no extra shots.
    """
    router = DLPRouter(init_logit=0.0, clamp=2.0, w_hw=w_hw,
                       lr=lr)

    log = {k: [] for k in [
        'dlp_fid', 'static_fid', 'greedy_fid',
        'fid_A', 'fid_B',
        'pA', 'pB', 'logit_A', 'logit_B',
        'selected', 'loss',
    ]}
    log.update({
        'backend': runner.name, 'shots': shots,
        'num_cycles': num_cycles,
        'noise_A': noise_A.tolist(),
        'noise_B': noise_B.tolist(),
        'lr': lr, 'n_grad_steps': n_grad_steps, 'w_hw': w_hw,
        'timestamp': datetime.now().isoformat(),
    })

    total_shots = 0
    t0 = time.time()

    hdr = (f"{'Cyc':>3} {'Sel':>3} {'DLP':>7} {'Static':>7} "
           f"{'Greedy':>7} {'F_A':>7} {'F_B':>7} "
           f"{'p(A)':>6} {'p(B)':>6} {'Loss':>7}")
    print(f"\n{hdr}\n{'-' * len(hdr)}")

    for t in range(num_cycles):
        nA, nB = noise_A[t], noise_B[t]

        # --- 1. snapshot router state BEFORE measuring ---
        p = router.probs()
        pA, pB = p[0], p[1]
        sel = router.selected_path()
        lA, lB = router.logits[0], router.logits[1]

        # --- 2. measure BOTH routes ---
        counts_A = runner.run(build_ghz_path_a(nA), shots)
        counts_B = runner.run(build_ghz_path_b(nB), shots)
        fid_A = ghz_fid(counts_A, shots)
        fid_B = ghz_fid(counts_B, shots)
        total_shots += 2 * shots

        # --- 3. deployed fidelity ---
        dlp_fid = fid_A if sel == 0 else fid_B

        # --- 4. gradient steps (paper: 10 per cycle with Adam) ---
        for _ in range(n_grad_steps):
            loss = router.update([fid_A, fid_B])

        # --- 5. baselines (reuse same measurements) ---
        static_fid = fid_A                    # always path A
        greedy_fid = max(fid_A, fid_B)        # oracle picks best

        # --- 6. log ---
        log['dlp_fid'].append(dlp_fid)
        log['static_fid'].append(static_fid)
        log['greedy_fid'].append(greedy_fid)
        log['fid_A'].append(fid_A)
        log['fid_B'].append(fid_B)
        log['pA'].append(float(pA))
        log['pB'].append(float(pB))
        log['logit_A'].append(float(lA))
        log['logit_B'].append(float(lB))
        log['selected'].append('A' if sel == 0 else 'B')
        log['loss'].append(float(loss))

        print(f"{t:3d}   {'A' if sel == 0 else 'B'}   "
              f"{dlp_fid:7.3f} {static_fid:7.3f} {greedy_fid:7.3f} "
              f"{fid_A:7.3f} {fid_B:7.3f} "
              f"{pA:6.3f} {pB:6.3f} {loss:7.4f}")

    log['elapsed_sec'] = time.time() - t0
    log['total_shots'] = total_shots
    log['final_probs'] = router.probs().tolist()
    log['final_logits'] = router.logits.tolist()
    return log

# ============================================================
#  Figures
# ============================================================

def plot_fidelity(log, filename):
    N = log['num_cycles']; c = np.arange(N)
    fig, ax = plt.subplots(figsize=(COLW, 2.2))

    ax.plot(c, log['fid_A'], ':', color='#4393C3', lw=0.8,
            alpha=0.5, label='Path A (meas.)')
    ax.plot(c, log['fid_B'], ':', color='#F4A582', lw=0.8,
            alpha=0.5, label='Path B (meas.)')
    ax.plot(c, log['static_fid'], '--', color='#D6604D', lw=1.0,
            alpha=0.8, label='Static (always A)')
    ax.plot(c, log['greedy_fid'], '-', color='0.55', lw=0.8,
            alpha=0.6, label='Greedy oracle')
    ax.plot(c, log['dlp_fid'], '-s', color='#2CA02C', lw=1.5,
            ms=4, zorder=5, label='DLP Router')

    ax.set_ylabel('GHZ Fidelity')
    ax.set_xlabel('Calibration Cycle')
    ax.set_ylim([0, 1.08]); ax.set_xlim([-0.3, N - 0.7])
    ax.set_xticks(c)
    ax.legend(loc='lower left', fontsize=6.5, ncol=2,
              framealpha=0.9, handlelength=1.8, columnspacing=1.0)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(f'{OUT}{filename}.{ext}')
    plt.close()


def plot_probs(log, filename):
    N = log['num_cycles']; c = np.arange(N)
    fig, ax = plt.subplots(figsize=(COLW, 1.5))

    ax.plot(c, log['pA'], '-o', color='#4393C3', lw=1.3, ms=3.5,
            label='$p_A$')
    ax.plot(c, log['pB'], '-o', color='#F4A582', lw=1.3, ms=3.5,
            label='$p_B$')
    ax.axhline(0.5, color='grey', ls=':', alpha=0.3, lw=0.5)

    ax.set_ylabel('Path Probability $p_i$')
    ax.set_xlabel('Calibration Cycle')
    ax.set_ylim([-0.05, 1.08]); ax.set_xlim([-0.3, N - 0.7])
    ax.set_xticks(c)
    ax.legend(loc='center right', fontsize=7, framealpha=0.9)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(f'{OUT}{filename}.{ext}')
    plt.close()


def plot_noise(nA, nB, filename):
    N = len(nA); c = np.arange(N)
    fig, ax = plt.subplots(figsize=(COLW, 1.3))
    ax.plot(c, nA, '-o', color='#4393C3', lw=1.2, ms=3,
            label='Path A noise')
    ax.plot(c, nB, '-o', color='#F4A582', lw=1.2, ms=3,
            label='Path B noise')
    ax.set_ylabel('$R_x$ noise (rad)')
    ax.set_xlabel('Cycle')
    ax.set_xlim([-0.3, N - 0.7]); ax.set_xticks(c)
    ax.legend(fontsize=7, framealpha=0.9)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(f'{OUT}{filename}.{ext}')
    plt.close()


def plot_combined(log, filename):
    N = log['num_cycles']; c = np.arange(N)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(DBLW, 3.2), sharex=True,
        gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.08})

    ax1.plot(c, log['fid_A'], ':', color='#4393C3', lw=0.8,
             alpha=0.5, label='Path A (meas.)')
    ax1.plot(c, log['fid_B'], ':', color='#F4A582', lw=0.8,
             alpha=0.5, label='Path B (meas.)')
    ax1.plot(c, log['static_fid'], '--', color='#D6604D', lw=1.0,
             alpha=0.8, label='Static (always A)')
    ax1.plot(c, log['greedy_fid'], '-', color='0.55', lw=0.8,
             alpha=0.6, label='Greedy oracle')
    ax1.plot(c, log['dlp_fid'], '-s', color='#2CA02C', lw=1.5,
             ms=4, zorder=5, label='DLP Router')
    ax1.set_ylabel('GHZ Fidelity')
    ax1.set_ylim([0, 1.08])
    ax1.legend(loc='lower left', fontsize=6.5, ncol=3,
               framealpha=0.9, handlelength=1.8, columnspacing=1.0)
    ax1.grid(True, alpha=0.25)

    ax2.plot(c, log['pA'], '-o', color='#4393C3', lw=1.3, ms=3.5,
             label='$p_A$')
    ax2.plot(c, log['pB'], '-o', color='#F4A582', lw=1.3, ms=3.5,
             label='$p_B$')
    ax2.axhline(0.5, color='grey', ls=':', alpha=0.3, lw=0.5)
    ax2.set_ylabel('$p_i$')
    ax2.set_xlabel('Calibration Cycle')
    ax2.set_ylim([-0.05, 1.08]); ax2.set_xticks(c)
    ax2.legend(loc='center right', fontsize=7, framealpha=0.9)
    ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(f'{OUT}{filename}.{ext}')
    plt.close()


def plot_logits(log, filename):
    N = log['num_cycles']; c = np.arange(N)
    fig, ax = plt.subplots(figsize=(COLW, 1.5))
    ax.plot(c, log['logit_A'], '-o', color='#4393C3', lw=1.3,
            ms=3.5, label='$\\lambda_A$')
    ax.plot(c, log['logit_B'], '-o', color='#F4A582', lw=1.3,
            ms=3.5, label='$\\lambda_B$')
    ax.axhline(0, color='grey', ls=':', alpha=0.3, lw=0.5)
    ax.set_ylabel('Structural Logit $\\lambda_i$')
    ax.set_xlabel('Calibration Cycle')
    ax.set_xlim([-0.3, N - 0.7]); ax.set_xticks(c)
    ax.legend(fontsize=7, framealpha=0.9)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(f'{OUT}{filename}.{ext}')
    plt.close()


def print_summary(log, label):
    dlp = np.mean(log['dlp_fid'])
    sta = np.mean(log['static_fid'])
    gre = np.mean(log['greedy_fid'])
    N = log['num_cycles']
    print(f"\n{'=' * 70}")
    print(f"RESULTS: {label} ({log['backend']})")
    print(f"{'=' * 70}")
    print(f"  Cycles:             {N}")
    print(f"  Shots/circuit:      {log['shots']}")
    print(f"  Total shots:        {log['total_shots']}  "
          f"(2 circuits/cycle, shared by all methods)")
    print(f"  Static (always A):  {sta:.4f}")
    print(f"  Greedy oracle:      {gre:.4f}")
    print(f"  DLP Router:         {dlp:.4f}")
    print(f"  DLP vs static:      {(dlp - sta) * 100:+.1f} pp")
    print(f"  DLP vs greedy:      {(dlp - gre) * 100:+.1f} pp")
    fp = log['final_probs']
    fl = log['final_logits']
    print(f"  Final probs:        p(A)={fp[0]:.4f}  p(B)={fp[1]:.4f}")
    print(f"  Final logits:       lA={fl[0]:.4f}  lB={fl[1]:.4f}")
    print(f"  Path selections:    {log['selected']}")
    print(f"  Time: {log['elapsed_sec']:.1f}s")
    print(f"{'=' * 70}")

# ============================================================
#  Main
# ============================================================
if __name__ == '__main__':
    t_total = time.time()

    api_key = load_api_key()
    hw = None
    if api_key:
        try:
            hw = IBMHardwareRunner(api_key, backend_name='ibm_fez')
        except Exception as e:
            print(f"Hardware connection failed: {e}")

    runner = hw if hw else NoiselessAerRunner()
    tag = 'hw' if hw else 'sim'
    shots = 512 if hw else 8192
    NUM_CYCLES = 12

    nA, nB = deterministic_crossing(NUM_CYCLES)

    print(f"\n{'=' * 70}")
    print(f"DLP SOFTMAX ROUTER — DETERMINISTIC CROSSING ({NUM_CYCLES} cycles)")
    print(f"  Runner:  {runner.name}")
    print(f"  Shots:   {shots}/circuit")
    print(f"  Routes:  A = cx(0,1)->cx(1,2)  |  B = cx(0,2)->cx(2,1)")
    print(f"  Loss:    L = sum p_i*(1-F_i) + w_hw*sum p_i*c_i")
    print(f"  Noise:   deterministic, no jitter")
    print(f"  Budget:  2 circuits/cycle, shared across all methods")
    print(f"{'=' * 70}")

    log = run_experiment(runner, nA, nB, num_cycles=NUM_CYCLES,
                         shots=shots, lr=0.5, n_grad_steps=10,
                         w_hw=0.0)

    plot_fidelity(log, f'fidelity_{tag}')
    plot_probs(log, f'probs_{tag}')
    plot_noise(nA, nB, f'noise_{tag}')
    plot_combined(log, f'combined_{tag}')
    plot_logits(log, f'logits_{tag}')

    print_summary(log, f"Deterministic Crossing ({tag})")

    with open(f'{OUT}results_{tag}.json', 'w') as f:
        json.dump(log, f, indent=2, default=str)

    elapsed = time.time() - t_total
    print(f"\nDONE -- {elapsed:.0f}s")
    print(f"Outputs: {OUT}")


DLP SOFTMAX ROUTER — DETERMINISTIC CROSSING (12 cycles)
  Runner:  aer_noiseless
  Shots:   8192/circuit
  Routes:  A = cx(0,1)->cx(1,2)  |  B = cx(0,2)->cx(2,1)
  Loss:    L = sum p_i*(1-F_i) + w_hw*sum p_i*c_i
  Noise:   deterministic, no jitter
  Budget:  2 circuits/cycle, shared across all methods

Cyc Sel     DLP  Static  Greedy     F_A     F_B   p(A)   p(B)    Loss
---------------------------------------------------------------------
  0   A     1.000   1.000   1.000   1.000   0.479  0.500  0.500  0.0094
  1   A     0.991   0.991   0.991   0.991   0.528  0.982  0.018  0.0177
  2   A     0.957   0.957   0.957   0.957   0.598  0.982  0.018  0.0493
  3   A     0.911   0.911   0.911   0.911   0.664  0.982  0.018  0.0934
  4   A     0.838   0.838   0.838   0.838   0.736  0.982  0.018  0.1642
  5   A     0.767   0.767   0.797   0.767   0.797  0.982  0.018  0.2326
  6   A     0.688   0.688   0.849   0.688   0.849  0.982  0.018  0.3081
  7   A     0.597   0.597   0.907   0.597   0.907  

/tmp/ipykernel_21547/3208986851.py:425: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



RESULTS: Deterministic Crossing (sim) (aer_noiseless)
  Cycles:             12
  Shots/circuit:      8192
  Total shots:        196608  (2 circuits/cycle, shared by all methods)
  Static (always A):  0.6959
  Greedy oracle:      0.9301
  DLP Router:         0.8883
  DLP vs static:      +19.2 pp
  DLP vs greedy:      -4.2 pp
  Final probs:        p(A)=0.0180  p(B)=0.9820
  Final logits:       lA=-2.0000  lB=2.0000
  Path selections:    ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B']
  Time: 5.0s

DONE -- 7s
Outputs: figures/b_experiment9_fix_crossing/


In [25]:
!rm -r figures